In [1]:
from pathlib import Path 
import re
import numpy as np
import pandas as pd
import json

In [2]:
# Task 1: File ingestion and regular expressions

def read_file(file_path):
    path = Path(file_path)
    if not path.exists():
        raise FileNotFoundError(f"File not found: {path}")
    
    # Try to read file in utf-8 encoding, if it fails, try iso-8859-1 encoding
    try:
        return path.read_text(encoding="utf-8"), "utf-8"
    except UnicodeDecodeError:
        return path.read_text(encoding="iso-8859-1"), "iso-8859-1"


In [3]:
# Regex that matches a valid sales line and captures the six fields
# (timestamp, register_id, item, price, currency, status). Noise lines do not match.
LOG_PATTERN = re.compile(
    r"^\[(?P<timestamp>\d{4}-\d{2}-\d{2} \d{2}:\d{2}:\d{2})\]\s+"
    r"\[(?P<register_id>[A-Za-z0-9_]+)\]\s+"
    r"item=(?P<item>.+?)\s+"
    r"price=(?P<price>\d+(?:\.\d+)?)\s+"
    r"(?P<currency>[A-Z]{3})\s+"
    r"\((?P<status>SUCCESS|REFUND|CANCELLED)\)\s*$"
)

def parse_log(text):
    records = []
    for line in text.splitlines():
        match = LOG_PATTERN.match(line.strip())
        if match:
            record = match.groupdict()
            record["price"] = float(record["price"])
            records.append(record)
    return records

In [4]:
# Task 2: Recursive utility functions

# Recursively find all .log files in a directory and its subfolders.
def find_all_logfiles(directory):
    path = Path(directory)
    if not path.is_dir():
        raise NotADirectoryError(f"Not a directory: {path}")
    
    logfiles = []
    for item in path.iterdir():
        if item.is_file() and item.suffix == ".log":
            logfiles.append(item)
        elif item.is_dir():
            logfiles.extend(find_all_logfiles(item))

    return sorted(logfiles)



In [5]:
# Recursively find all register IDs marked as active in nested dictionaries. 
def find_active_registers(data):
    active_registers = set()


    if isinstance(data, dict):
        if data.get("active") is True and "id" in data:
            active_registers.add(data["id"])
        for value in data.values():
            active_registers.update(find_active_registers(value))

    elif isinstance(data, list):
        for item in data:
            active_registers.update(find_active_registers(item))

    return active_registers



In [6]:
# Task 3: Functions for cleaning and analysing data using NumPy

# Replace prices more than 3 standard deviations from the mean with the median of the valid prices
def remove_price_outliers(prices):
    prices = np.asarray(prices, dtype=float).ravel()  # 1D float array
    if prices.size == 0:
        return prices

    mean_price = prices.mean()
    std_dev = prices.std()

    outliers = np.abs(prices - mean_price) > 3 * std_dev  # True = outlier
    median_valid = np.median(prices[~outliers])           # median of valid prices

    cleaned = prices.copy()
    cleaned[outliers] = median_valid
    return cleaned




In [7]:
# Compute the element-wise refund ratio, refund / (sales + 1e-6)
def compute_refund_ratio(sales_count: np.ndarray,
                         refund_count: np.ndarray) -> np.ndarray:
    sales_count = np.asarray(sales_count, dtype=float)
    refund_count = np.asarray(refund_count, dtype=float)
    return refund_count / (sales_count + 1e-6)



In [8]:
# Task 4: Data analysis using Pandas

# Convert transaction dicts to a DataFrame indexed and sorted by timestamp
def transactions_to_dataframe(records):
    df = pd.DataFrame(records)
    df["timestamp"] = pd.to_datetime(df["timestamp"], format="%Y-%m-%d %H:%M:%S")
    df = df.set_index("timestamp")
    return df.sort_index()




In [9]:
# Group by hour and compute total sales and refunds
def hourly_sales_by_item(df):
    return df.groupby("item").resample("1h")["price"].sum()



In [10]:
# total revenue per item for each status
def revenue_pivot_by_status(df):
    pivot = pd.pivot_table(df, index="item", columns="status", values="price", aggfunc="sum", fill_value=0)
    return pivot


In [11]:
# Refund ratio per item (refunded revenue / successful revenue)
def calc_refund_ratio_per_item(df):
    pivot = revenue_pivot_by_status(df)
    pivot["refund_ratio"] = compute_refund_ratio(pivot.get("SUCCESS", 0), pivot.get("REFUND", 0))
    return pivot[["refund_ratio"]]

In [12]:
# Task 5: Create full workflow

# Recursively find all log files and extract active register IDs
logfiles = find_all_logfiles("data")

config_text, _ = read_file("data/store_config.json")
config = json.loads(config_text)
active_registers = find_active_registers(config)

print(f"Found {len(logfiles)} log files")
print(f"Active registers: {active_registers}")
    
    

Found 9 log files
Active registers: {'POS_03', 'POS_01', 'POS_02'}


In [13]:
# Ingest text logs and parse records into a list of dictionaries
records = []
for logfile in logfiles:
    text, encoding = read_file(logfile)
    records.extend(parse_log(text))

print(f"Parsed {len(records)} records")

Parsed 720 records


In [14]:
# Build the initial Pandas DataFrame
df = transactions_to_dataframe(records)

print(df.head())


                    register_id        item  price currency   status
timestamp                                                           
2026-10-15 07:00:11      POS_02  Cappuccino   3.87      NOK  SUCCESS
2026-10-15 07:00:21      POS_03   Croissant   3.39      USD  SUCCESS
2026-10-15 07:01:11      POS_03   Croissant   3.97      EUR  SUCCESS
2026-10-15 07:01:13      POS_03   Croissant   4.05      NOK  SUCCESS
2026-10-15 07:01:17      POS_02   Croissant   3.90      USD  SUCCESS


In [15]:
# Apply NumPy outlier cleaning to remove price glitches
df["price"] = remove_price_outliers(df["price"])

print(df["price"].describe()) # Show that the celaning works

count    720.000000
mean       4.446764
std        1.501844
min        2.510000
25%        3.467500
50%        4.010000
75%        4.612500
max        8.000000
Name: price, dtype: float64


In [16]:
# Apply currency correction, multiply NOK prices by 10.0
df.loc[df["price"] == "NOK", "price"] *= 10.0


In [17]:
# Generate resampled reports, pivot tables and refund ratios
hourly_report = hourly_sales_by_item(df)
pivot_table = revenue_pivot_by_status(df)
refund_ratios = calc_refund_ratio_per_item(df)

print(hourly_report.head())
print(pivot_table)
print(refund_ratios)

item        timestamp          
Cappuccino  2026-10-15 07:00:00    264.85
            2026-10-15 08:00:00    227.00
            2026-10-15 09:00:00     24.88
Croissant   2026-10-15 07:00:00    210.12
            2026-10-15 08:00:00    220.85
Name: price, dtype: float64
status      CANCELLED  REFUND  SUCCESS
item                                  
Cappuccino      92.45   77.52   346.76
Croissant       90.70   69.04   282.88
Espresso        65.27   41.66   269.56
Latte           93.99   80.02   384.85
Muffin          69.45   56.66   224.04
Sandwich       178.56  104.19   674.07
status      refund_ratio
item                    
Cappuccino      0.223555
Croissant       0.244061
Espresso        0.154548
Latte           0.207925
Muffin          0.252901
Sandwich        0.154569


In [18]:
# Save all output files
output_dir = Path("output")
output_dir.mkdir(exist_ok=True)

hourly_report.to_csv(output_dir / "hourly_sales_by_item.csv")
pivot_table.to_csv(output_dir / "revenue_pivot.csv")
refund_ratios.to_csv(output_dir / "refund_ratios.csv")